# lambda_reg (L1) sweep (stride_x=stride_t=10)

Sweeps `lam_reg ? {1e-5, 1e-3, 0.1, 1, 5}` under fixed controls.

**Controls**
- `stride_x = 10`, `stride_t = 10`
- `noise=0.7`, `nu=0.02`
- `steps=8000`, `batch_size=1000`, `lr=0.001`
- `lam_pde=0.5`, `lam_data=10`, `lam_tv=0`

**Dependents (same table outputs)**
- Final `loss_data`, `loss_pde`
- `w_u`, `w_ux`, `w_uxx`, `w_prod` from `eql.readout.weight`
- `M01`, `M10` from `eql.effective_quadratic_matrix(symmetrize=False)`


In [2]:
import sys

sys.path.append('..')  # add project root

import numpy as np
import torch
import matplotlib.pyplot as plt

from prog import mlps, featlib, trainer, hlprs
import Datasets.matconv as mc

SEED = 1432
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cpu')


In [ ]:
# Controls

steps = 8000
log_every = 1000

noise = 0.7
nu = 0.02

stride_x = 10
stride_t = 10

lr = 1e-3
batch_size = 1000

lam_pde = 1.0
lam_data = 1.0
lam_tv = 0.0

selected_derivs = ('u','u_x','u_xx')

lam_reg_values = [0, 1e-6, 5e-6, 1e-5, 5e-5]

# Dataset partitioning controls
part_num = 1
which_part = 1


In [4]:
# Build dataset (fixed stride_x=stride_t=10)

partitions = mc.build_dataset_from_burgers(
    noise_level=noise,
    nu=nu,
    stride_t=stride_t,
    stride_x=stride_x,
    seed=SEED,
    quantile_splits=part_num,
    return_partitions=True,
)

key = [k for k in partitions if k.startswith(f'Q{which_part}:')][0]
t_np, x_np, y_np, y_noisy_np, _N = partitions[key]

# torch tensors

t_torch       = torch.from_numpy(t_np).to(device)
x_torch       = torch.from_numpy(x_np).to(device)
y_clean_torch = torch.from_numpy(y_np).to(device)
y_noisy_torch = torch.from_numpy(y_noisy_np).to(device)

print('N points:', t_np.shape[0])
print('x_unique:', len(np.unique(x_np)), 't_unique:', len(np.unique(t_np)))


N points: 676
x_unique: 26 t_unique: 26


In [ ]:
def run_once(lam_reg: float):
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    u_model = mlps.SimpleMLP(n_layers=4, hidden_size=64, act=mlps.Sin)
    symnet  = mlps.EQL(in_dim=len(selected_derivs), prod_dim=2, num_layers=1, bias=False)

    train_config = trainer.TrainerConfig(
        lr=lr,
        lambda_pde=lam_pde,
        lambda_reg=lam_reg,
        lambda_tv=lam_tv,
        lambda_data=lam_data,
        selected_derivs=selected_derivs,
        device=device,
    )

    ft = featlib.FeatureTensor(selected_derivs, normalize=False)
    feature_builder = ft.build

    train = trainer.PDETrainer(
        u_model=u_model,
        v_model=symnet,
        cfg=train_config,
        feature_builder=feature_builder,
    )

    loss_dat = []
    loss_pde = []

    for i in range(steps):
        t, x, u_noisy, u_clean = hlprs.make_batch(
            batch_size=batch_size,
            t_torch=t_torch,
            x_torch=x_torch,
            y_clean=y_clean_torch,
            y_noisy=y_noisy_torch,
        )
        out = train.step(t=t, x=x, u_noisy=u_noisy, u_clean=u_clean)
        loss_dat.append(out['loss_data'])
        loss_pde.append(out['loss_pde'])

        if log_every and i % log_every == 0:
            print(f"[lam_reg={lam_reg:g}] step {i}: data={out['loss_data']:.6e} pde={out['loss_pde']:.6e} l1={out['l1']:.6e}")

    w = symnet.readout.weight.detach().cpu().numpy().reshape(-1)
    M = symnet.effective_quadratic_matrix(symmetrize=False).detach().cpu().numpy()
    print(hlprs.snapshot_comp(train.u, stride_x, stride_t, y_noisy_np, y_np, t_np, x_np, snap_no=10)[:0])

    return {
        'lam_reg': float(lam_reg),
        'loss_data_final': float(loss_dat[-1]),
        'loss_pde_final': float(loss_pde[-1]),
        'w_u': float(w[0]),
        'w_ux': float(w[1]),
        'w_uxx': float(w[2]),
        'w_prod': float(w[-1]),
        'M01': float(M[0,1]),
        'M10': float(M[1,0]),
        'l1_final': float(out['l1']),
        'loss_data_curve': loss_dat,
        'loss_pde_curve': loss_pde,
        'readout_weight': w,
        'M_eff': M,
    }


In [7]:
# Run sweep

results = []
for lam_reg in lam_reg_values:
    print('==============================')
    print('Running lam_reg =', lam_reg)
    print('==============================')
    results.append(run_once(lam_reg))


Running lam_reg = 1e-05
[lam_reg=1e-05] step 0: data=1.003832e+00 pde=7.057870e-03 l1=3.025851e+00
[lam_reg=1e-05] step 1000: data=5.311525e-01 pde=6.550155e-04 l1=2.889308e+00
[lam_reg=1e-05] step 2000: data=5.440436e-01 pde=1.638429e-03 l1=2.994641e+00
[lam_reg=1e-05] step 3000: data=4.845819e-01 pde=1.114474e-03 l1=2.992554e+00
[lam_reg=1e-05] step 4000: data=5.007139e-01 pde=1.057396e-03 l1=3.013924e+00
[lam_reg=1e-05] step 5000: data=4.769658e-01 pde=1.158216e-03 l1=3.071706e+00
[lam_reg=1e-05] step 6000: data=5.170031e-01 pde=1.786413e-03 l1=3.134135e+00
[lam_reg=1e-05] step 7000: data=4.785875e-01 pde=2.500728e-03 l1=3.292359e+00
Running lam_reg = 0.001
[lam_reg=0.001] step 0: data=1.003832e+00 pde=7.057870e-03 l1=3.025851e+00
[lam_reg=0.001] step 1000: data=5.321282e-01 pde=1.036627e-03 l1=2.121364e-01
[lam_reg=0.001] step 2000: data=5.445532e-01 pde=7.207928e-04 l1=1.040503e-01
[lam_reg=0.001] step 3000: data=4.849427e-01 pde=6.579197e-04 l1=1.255067e-01
[lam_reg=0.001] step 4

In [8]:
# Table output

try:
    import pandas as pd
except ImportError:
    pd = None

cols = [
    'lam_reg',
    'loss_data_final','loss_pde_final',
    'w_u','w_ux','w_uxx','w_prod',
    'M01','M10',
]

rows = [{k: r[k] for k in cols} for r in results]

if pd is not None:
    df = pd.DataFrame(rows).sort_values(['lam_reg']).reset_index(drop=True)
    display(df)
    print('Markdown table (paste into email):')
    print(df.to_markdown(index=False))
else:
    print(rows)


,lam_reg,loss_data_final,loss_pde_final,w_u,w_ux,w_uxx,w_prod,M01,M10
0,0.00001,0.485701,0.002622,-0.126251,0.024440,-0.003843,-0.567633,-5.817191e-01,2.523771e-03
1,0.00100,0.484741,0.001156,-0.080522,-0.081048,-0.001116,0.000062,1.942361e-13,-2.719334e-13
2,0.10000,0.486759,0.001276,-0.000085,0.000419,0.000013,0.000035,-4.445333e-14,3.516891e-13
3,1.00000,0.487004,0.001326,0.000193,-0.000042,-0.000077,-0.000072,-1.178554e-12,1.290641e-14
4,5.00000,0.486998,0.001292,-0.000190,-0.000189,0.000304,-0.000087,1.285398e-12,-3.196447e-12


Markdown table (paste into email):


ImportError: Missing optional dependency 'tabulate'.  Use pip or conda to install tabulate.

In [ ]:
# Plot loss curves (data, pde) per lam_reg

plt.figure(figsize=(8,4))
for r in results:
    plt.plot(r['loss_data_curve'], label=f"data lam_reg={r['lam_reg']:g}", alpha=0.8)
plt.yscale('log'); plt.xlabel('step'); plt.ylabel('loss'); plt.title('Data loss'); plt.legend(fontsize=7, ncols=2); plt.tight_layout(); plt.show()

plt.figure(figsize=(8,4))
for r in results:
    plt.plot(r['loss_pde_curve'], label=f"pde lam_reg={r['lam_reg']:g}", alpha=0.8)
plt.yscale('log'); plt.xlabel('step'); plt.ylabel('loss'); plt.title('PDE loss'); plt.legend(fontsize=7, ncols=2); plt.tight_layout(); plt.show()
